In [ ]:
import pandas as pd
import os

# -------------------------
# PATH SETUP
# -------------------------

BASE_DIR = os.path.dirname(os.path.abspath(__file__))   # /notebooks
PROJECT_DIR = os.path.dirname(BASE_DIR)                 # project root
OUTPUT_DIR = os.path.join(PROJECT_DIR, "data", "clean")
INPUT_FILE = os.path.join(PROJECT_DIR, "data", "raw.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# -------------------------
# LOAD DATA
# -------------------------
df = pd.read_csv(INPUT_FILE)

# -------------------------
# HELPERS
# -------------------------
def split_list(x):
    if pd.isna(x):
        return []
    return [i.strip() for i in str(x).split(",") if i.strip()]

# -------------------------
# CONTAINERS
# -------------------------
products = []
users = {}
reviews = []
images = []
product_categories = []

# -------------------------
# MAIN LOOP
# -------------------------
for _, row in df.iterrows():

    pid = row["product_id"]

    # PRODUCTS
    products.append({
        "product_id": pid,
        "product_name": row["product_name"],
        "about_product": row["about_product"],
        "discounted_price": row["discounted_price"],
        "actual_price": row["actual_price"],
        "discount_percentage": row["discount_percentage"],
        "rating": row["rating"],
        "rating_count": row["rating_count"],
        "product_link": row["product_link"]
    })

    # USERS + REVIEWS
    user_ids = split_list(row["user_id"])
    user_names = split_list(row["user_name"])

    review_ids = split_list(row["review_id"])
    titles = split_list(row["review_title"])
    contents = split_list(row["review_content"])

    n = min(len(review_ids), len(titles), len(contents), len(user_ids))

    for i in range(n):

        uid = user_ids[i]
        uname = user_names[i] if i < len(user_names) else None

        if uid not in users:
            users[uid] = {
                "user_id": uid,
                "user_name": uname
            }

        reviews.append({
            "review_id": review_ids[i],
            "product_id": pid,
            "user_id": uid,
            "review_title": titles[i],
            "review_content": contents[i]
        })

    # IMAGES
    imgs = split_list(row["img_link"])
    for img in imgs:
        images.append({
            "product_id": pid,
            "image_url": img
        })

    # CATEGORIES (raw, no processing)
    product_categories.append({
        "product_id": pid,
        "category_name": row["category"]
    })

# -------------------------
# EXPORT CSVs
# -------------------------
# pd.DataFrame(products).to_csv(os.path.join(OUTPUT_DIR, "products.csv"), index=False)
# pd.DataFrame(list(users.values())).to_csv(os.path.join(OUTPUT_DIR, "users.csv"), index=False)
# pd.DataFrame(reviews).to_csv(os.path.join(OUTPUT_DIR, "reviews.csv"), index=False)
# pd.DataFrame(images).to_csv(os.path.join(OUTPUT_DIR, "product_images.csv"), index=False)
# pd.DataFrame(product_categories).to_csv(os.path.join(OUTPUT_DIR, "product_categories.csv"), index=False)

print(f"Done! Files saved to: {OUTPUT_DIR}")